# Module 48: Custom Autograd Functions

Define custom `forward` / `backward` with `torch.autograd.Function`.

**Topics:**
- Subclassing `Function` and calling `.apply`
- `ctx.save_for_backward` vs metadata on `ctx`
- Straight-through estimators
- `gradcheck` and double backward with `create_graph=True`

In [ ]:
import torch
from torch.autograd import Function, grad, gradcheck

class Square(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * x

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x

x = torch.tensor(3.0, requires_grad=True)
y = Square.apply(x)
y.backward()
print(f"y={y.item()}, x.grad={x.grad.item()}")  # expect 6.0

In [ ]:
class LogSumExp(Function):
    @staticmethod
    def forward(ctx, x):
        x_max = x.max(dim=-1, keepdim=True).values
        exp_shifted = (x - x_max).exp()
        sum_exp = exp_shifted.sum(dim=-1, keepdim=True)
        ctx.save_for_backward(exp_shifted, sum_exp)
        return (sum_exp.log() + x_max).squeeze(-1)

    @staticmethod
    def backward(ctx, grad_output):
        exp_shifted, sum_exp = ctx.saved_tensors
        return grad_output.unsqueeze(-1) * (exp_shifted / sum_exp)

t = torch.randn(2, 5, dtype=torch.double, requires_grad=True)
print("gradcheck:", gradcheck(LogSumExp.apply, (t,), eps=1e-6, atol=1e-4))

In [ ]:
class BinarySTE(Function):
    @staticmethod
    def forward(ctx, x):
        return (x > 0).to(dtype=x.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output  # identity STE

z = torch.tensor([-2.0, -0.1, 0.3, 1.5], requires_grad=True)
out = BinarySTE.apply(z)
out.sum().backward()
print("binary forward:", out.tolist())
print("STE grads:", z.grad.tolist())

In [ ]:
class SoftPlus(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.where(x > 20, x, torch.log1p(torch.exp(torch.clamp(x, max=20))))

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * torch.sigmoid(x)

w = torch.tensor(1.5, requires_grad=True)
loss = SoftPlus.apply(w)
(g,) = grad(loss, w, create_graph=True)
(h,) = grad(g, w)
print(f"softplus'(1.5)={g.item():.4f}, softplus''(1.5)={h.item():.4f}")

## Next Steps

- Run `autograd_function_basics.py` and `double_backward.py` in this module
- Read [Extending PyTorch](https://pytorch.org/docs/stable/notes/extending.html)
- Continue to Module 49 for advanced activation checkpointing